# Week 01 — Hugging Face basics

Chọn kernel `llm-compression-lab`, rồi **Restart Kernel → Run All**.
Notebook đọc cấu hình, tokenize dataset các prompt, tải model một lần, sinh và lưu kết quả cho từng prompt.
`config["prompt"]` chọn ví dụ để quan sát token/logits; danh sách dataset quyết định các prompt thực nghiệm.
Kết quả mới nằm trong `outputs/week01/run_<timestamp>/`. Các kết quả cũ được giữ nguyên.
Lần đầu cần Internet để tải model; khảo sát Hub có thể được bỏ qua tự động nếu kết nối lỗi.

In [74]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
config_path = ROOT / "configs" / "week01.json"
if not config_path.is_file():
    raise FileNotFoundError("Open Jupyter from the project root or notebooks directory")
config = json.loads(config_path.read_text(encoding="utf-8"))
if not config["model_id"]:
    raise ValueError("Set model_id in configs/week01.json before continuing")
config

{'model_id': 'gpt2',
 'revision': '607a30d783dfa663caf39e06633721c8d4cfcd7e',
 'device': 'cuda',
 'prompt': 'The purpose of a university is',
 'max_new_tokens': 50,
 'seed': 42}

## 1. Môi trường
Ghi lại phiên bản thư viện, thiết bị và seed vào nhật ký.

In [75]:
import sys
print(sys.version)
print(sys.executable)

3.11.18 | packaged by Anaconda, Inc. | (main, Oct 26 2026, 18:13:29) [MSC v.1942 64 bit (AMD64)]
d:\IDEs\Miniconda3\envs\llm-compression-lab\python.exe


In [76]:
import importlib.metadata
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

for name in ["torch", "transformers", "datasets", "tokenizers", "safetensors"]:
    print(name, importlib.metadata.version(name))
print("CUDA:", torch.cuda.is_available())
set_seed(config["seed"])
device = config["device"]
if device == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; use cpu or configure CUDA-enabled PyTorch")

torch 2.9.1+cu126
transformers 5.23.0
datasets 5.2.0
tokenizers 0.23.3
safetensors 0.8.0
CUDA: True


## 2. Tokenizer
Lần đầu cần Internet. Quan sát token IDs, attention mask và kết quả giải mã.

In [77]:
tokenizer = AutoTokenizer.from_pretrained(
    config["model_id"],
    revision=config["revision"]
)
encoded = tokenizer(config["prompt"], return_tensors="pt")
print(encoded)
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"][0].tolist()))
print(tokenizer.decode(encoded["input_ids"][0]))

{'input_ids': tensor([[ 464, 4007,  286,  257, 6403,  318]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}
['The', 'Ġpurpose', 'Ġof', 'Ġa', 'Ġuniversity', 'Ġis']
The purpose of a university is


In [78]:
input_ids = encoded["input_ids"]
attention_mask = encoded["attention_mask"]

print("Input shape:", tuple(input_ids.shape))
print("Token IDs:", input_ids[0].tolist())
print("Tokens:", tokenizer.convert_ids_to_tokens(input_ids[0].tolist()))
print("Attention mask:", attention_mask.tolist())
print("Decoded text:", tokenizer.decode(input_ids[0]))

Input shape: (1, 6)
Token IDs: [464, 4007, 286, 257, 6403, 318]
Tokens: ['The', 'Ġpurpose', 'Ġof', 'Ġa', 'Ġuniversity', 'Ġis']
Attention mask: [[1, 1, 1, 1, 1, 1]]
Decoded text: The purpose of a university is


## 2a. Dataset thực nghiệm
`map` thêm token IDs và attention mask vào mỗi dòng. Inference bên dưới dùng trực tiếp các dòng đã tokenize, lần lượt từng prompt để chưa cần padding batch.

In [79]:
from datasets import Dataset

prompt_dataset = Dataset.from_dict({
    "text": [
        "Machine learning is",
        "Once upon a time,",
        "The purpose of a university is",
        "The capital of Vietnam is",
        "The best university in Vietnam is",
    ]
})
tokenized_dataset = prompt_dataset.map(
    lambda batch: tokenizer(batch["text"]), batched=True,
)
print("Original dataset:", prompt_dataset)
print("Tokenized columns:", tokenized_dataset.column_names)
print("First example:", tokenized_dataset[0])

Map: 100%|██████████| 5/5 [00:00<00:00, 1662.56 examples/s]

Original dataset: Dataset({
    features: ['text'],
    num_rows: 5
})
Tokenized columns: ['text', 'input_ids', 'attention_mask']
First example: {'text': 'Machine learning is', 'input_ids': [37573, 4673, 318], 'attention_mask': [1, 1, 1]}


## 3. Model và inference
Dùng greedy decoding để dễ so sánh. Khung này dành cho causal language model; mô hình chat có thể cần chat template riêng.

In [85]:
model = AutoModelForCausalLM.from_pretrained(
    config["model_id"], revision=config["revision"],
)
model = model.to(device).eval()
pad_token_id = tokenizer.pad_token_id
if pad_token_id is None:
    pad_token_id = tokenizer.eos_token_id

results = []
for row in tokenized_dataset:
    set_seed(config["seed"])
    row_inputs = {
        key: torch.tensor([row[key]], dtype=torch.long, device=device)
        for key in ["input_ids", "attention_mask"]
    }
    with torch.inference_mode():
        row_output_ids = model.generate(
            **row_inputs,
            max_new_tokens=config["max_new_tokens"],
            do_sample=False,
            pad_token_id=pad_token_id,
        )
    prompt_length = row_inputs["input_ids"].shape[1]
    row_new_ids = row_output_ids[0, prompt_length:]
    result = {
        "config": {**config, "prompt": row["text"]},
        "requested_revision": config["revision"],
        "model_revision": getattr(model.config, "_commit_hash", None),
        "generated_text": tokenizer.decode(row_output_ids[0], skip_special_tokens=True),
        "continuation": tokenizer.decode(row_new_ids, skip_special_tokens=True),
        "new_token_count": row_new_ids.numel(),
        "first_generated_token_id": row_new_ids[0].item() if row_new_ids.numel() else None,
        "do_sample": False,
        "device": str(next(model.parameters()).device),
        "dtype": str(next(model.parameters()).dtype),
        "python": sys.version,
        "versions": {
            name: importlib.metadata.version(name)
            for name in ["torch", "transformers", "datasets", "tokenizers", "safetensors", "huggingface-hub"]
        },
    }
    results.append(result)
    print("Prompt:", row["text"])
    print("Continuation:", result["continuation"])
    print("New token count:", result["new_token_count"])
    print()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 472.15it/s]


Prompt: Machine learning is
Continuation:  a very powerful tool for learning about the world around us. It's a tool that can help us understand the world around us, and it's a tool that can help us understand the world around us.

The world is changing. We're
New token count: 50

Prompt: Once upon a time,
Continuation:  the world was a place of great beauty and great danger. The world was a place of great danger, and the world was a place of great danger. The world was a place of great danger, and the world was a place of great danger.
New token count: 50

Prompt: The purpose of a university is
Continuation:  to provide a safe environment for students to learn and to develop their skills.

"We are committed to providing a safe environment for students to learn and to develop their skills.

"We are committed to providing a safe environment for students to
New token count: 50

Prompt: The capital of Vietnam is
Continuation:  the capital of the Republic of Vietnam. The capital of the Repu

In [86]:
# Select one dataset result for the learning cells below.
example_result = next(
    (item for item in results if item["config"]["prompt"] == config["prompt"]),
    results[0],
)
example_encoded = tokenizer(example_result["config"]["prompt"], return_tensors="pt")
inputs = {key: value.to(device) for key, value in example_encoded.items()}
print("Observed prompt:", example_result["config"]["prompt"])
print("Continuation:", example_result["continuation"])
print("Model device:", example_result["device"])
print("Model dtype:", example_result["dtype"])

Observed prompt: The purpose of a university is
Continuation:  to provide a safe environment for students to learn and to develop their skills.

"We are committed to providing a safe environment for students to learn and to develop their skills.

"We are committed to providing a safe environment for students to
Model device: cuda:0
Model dtype: torch.float32


In [87]:
with torch.inference_mode():
    forward_output = model(**inputs)
logits = forward_output.logits
next_token_id = logits[0, -1].argmax().item()
print("Logits shape:", tuple(logits.shape))
print("Next token ID:", next_token_id)
print("Next token text:", repr(tokenizer.decode([next_token_id])))
print("First generated token ID:", example_result["first_generated_token_id"])

Logits shape: (1, 6, 50257)
Next token ID: 284
Next token text: ' to'
First generated token ID: 284


## 4. Lưu toàn bộ kết quả
Mỗi Run All tạo một thư mục riêng với một JSON cho mỗi prompt và bảng tổng hợp. Không sửa lịch sử cũ. `requested_revision` là revision truyền khi tải; `model_revision` vẫn có thể null nếu thư viện không cung cấp thuộc tính nội bộ.

In [88]:
from datetime import datetime, timezone
import csv

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = ROOT / "outputs" / "week01" / f"run_{run_id}"
output_dir.mkdir(parents=True, exist_ok=False)
summary_rows = []
for index, result in enumerate(results, start=1):
    output_path = output_dir / f"inference_{index:02d}.json"
    output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    summary_rows.append({
        "prompt": result["config"]["prompt"],
        "new_token_count": result["new_token_count"],
        "continuation": result["continuation"],
        "output_file": output_path.name,
    })
with (output_dir / "summary.csv").open("w", encoding="utf-8-sig", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary_rows[0]))
    writer.writeheader()
    writer.writerows(summary_rows)
print("Saved results:", len(results))
print("Output directory:", output_dir)

Saved results: 5
Output directory: D:\SourceCodes\Python_SourceCodes\llm-compression-lab\outputs\week01\run_20261115T104944859289Z


## 5. Khảo sát file trên Hugging Face
Truy vấn đúng revision trong cấu hình, không tải toàn bộ repository. Nếu mạng lỗi, kết quả inference phía trên vẫn đã được lưu.

In [89]:
from huggingface_hub import HfApi

api = HfApi()
try:
    info = api.model_info(config["model_id"], revision=config["revision"])
    print("Requested repository revision:", info.sha)
    for filename in api.list_repo_files(config["model_id"], revision=config["revision"]):
        print(filename)
except Exception as exc:
    print("Hub inspection skipped:", type(exc).__name__, str(exc))
    print("Inference results have already been saved. Retry this cell when online.")

Requested repository revision: 607a30d783dfa663caf39e06633721c8d4cfcd7e
.gitattributes
64-8bits.tflite
64-fp16.tflite
64.tflite
README.md
config.json
flax_model.msgpack
generation_config.json
merges.txt
model.safetensors
onnx/config.json
onnx/decoder_model.onnx
onnx/decoder_model_merged.onnx
onnx/decoder_with_past_model.onnx
onnx/generation_config.json
onnx/merges.txt
onnx/special_tokens_map.json
onnx/tokenizer.json
onnx/tokenizer_config.json
onnx/vocab.json
pytorch_model.bin
rust_model.ot
tf_model.h5
tokenizer.json
tokenizer_config.json
vocab.json


## 6. Khảo sát trọng số cục bộ
Chỉ tìm file trong cache của revision đã yêu cầu và đọc tên/shape tensor, không tải thêm file. Đây không phải phép đo VRAM hay số tham số trainable.

In [ ]:
from huggingface_hub import constants, try_to_load_from_cache
from safetensors import safe_open

print("Hub cache:", constants.HF_HUB_CACHE)
weight_file = try_to_load_from_cache(
    config["model_id"], "model.safetensors", revision=config["revision"],
)
if isinstance(weight_file, str):
    weight_path = Path(weight_file)
    print("Weight file:", weight_path)
    print("Weight bytes:", weight_path.stat().st_size)
    with safe_open(str(weight_path), framework="pt", device="cpu") as handle:
        names = list(handle.keys())
        print("Stored tensor count:", len(names))
        for name in names[:10]:
            print(name, "->", handle.get_slice(name).get_shape())
else:
    print("No single model.safetensors file found for this revision in this cache.")

## 7. Ghi chú sau thực hành

- Đọc `summary.csv` và các JSON trong thư mục kết quả được in ở mục 4.
- Nhận xét độ liên quan, lặp từ/ý và câu bị ngắt; không coi ba prompt là đánh giá chất lượng tổng quát.
- Hoàn thiện `docs/model_files.md`, `docs/experiment_log.md` và báo cáo.
- Sau Restart Kernel → Run All, lưu notebook bằng Ctrl+S để giữ output.
- Model card và nhật ký vẫn cần bạn đọc/viết; Run All tự động hóa phần tính toán và lưu kết quả.